<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Mobilnosc_calkowita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza Całkowitej Mobilności Miejskiej w Krakowie (Auta + GTFS-RT)

Niniejszy notatnik służy do kompleksowej analizy natężenia ruchu i opóźnień w Krakowie, łącząc zjawiska drogowe dotyczące samochodów osobowych z opóźnieniami w komunikacji miejskiej.

Wykorzystujemy połączone zbiory danych:
- **Historię zatłoczenia drogowego** (dane z plików CSV o punktach pomiarowych).
- **Komunikację Miejską (GTFS-RT ZTP)** (format Protobuf) dla autobusów i tramwajów pobieraną w czasie rzeczywistym.

Notatnik agreguje oba rodzaje ruchu na spójnej siatce drogowej pochodzącej z systemu **OSMnx** i generuje interaktywne mapy ulic ukazujące całościowe natężenie ruchu (wspólne korki), a także globalne statystyki najgorszych godzin i skrzyżowań.

In [ ]:
# Jeśli nie masz ich zainstalowanych, odkomentuj i uruchom poniższą linijkę:
!pip install gtfs-realtime-bindings protobuf requests pandas plotly folium scipy osmnx networkx matplotlib
import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import TimestampedGeoJson
import osmnx as ox
import networkx as nx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import datetime
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import time

def fetch_gtfs_rt_positions():
    print("Pobieranie danych GTFS-RT (VehiclePositions) dla Krakowa...")

    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/VehiclePositions_T.pb",
        "Autobusy": "https://gtfs.ztp.krakow.pl/VehiclePositions_A.pb"
    }

    positions_data = []

    for v_type, url in urls.items():
        print(f" Pobieranie pozycji dla: {v_type}...")
        
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = requests.get(url, timeout=15)
                response.raise_for_status()
                
                if response.content.startswith(b'<'):
                    raise ValueError("Otrzymano dokument HTML zamiast surowego protokołu Protobuf (błąd serwera ZTP).")

                feed = gtfs_realtime_pb2.FeedMessage()
                feed.ParseFromString(response.content)
                
                for entity in feed.entity:
                    if entity.HasField('vehicle') and entity.vehicle.HasField('position'):
                        veh = entity.vehicle
                        lat = veh.position.latitude
                        lon = veh.position.longitude
                        
                        route_id = None
                        if veh.HasField('trip') and veh.trip.HasField('route_id'):
                            route_id = veh.trip.route_id
                            
                        timestamp = None
                        if veh.HasField('timestamp'):
                            timestamp = veh.timestamp
                            
                        positions_data.append({
                            "typ": v_type,
                            "line_num": route_id,
                            "lat": lat,
                            "lon": lon,
                            "time": timestamp
                        })
                break
            except Exception as e:
                print(f"  Błąd podczas pobierania {v_type} (próba {attempt+1}/{max_retries}): {e}")
                if attempt < max_retries - 1:
                    time.sleep(2)

    df_pos = pd.DataFrame(positions_data)
    
    if not df_pos.empty:
        if 'time' in df_pos.columns:
            df_pos['datetime'] = pd.to_datetime(df_pos['time'].fillna(datetime.datetime.now().timestamp()), unit='s')
            df_pos['hour'] = df_pos['datetime'].dt.strftime('%Y-%m-%dT%H:00:00')
        else:
            df_pos['hour'] = datetime.datetime.now().strftime('%Y-%m-%dT%H:00:00')
            
        print(f"\nZakończono. Pobrano {len(df_pos)} współrzędnych pojazdów komunikacji miejskiej.")
    else:
        print("\nNie udało się pobrać żadnych pozycji lub brak pojazdów na trasie.")
        
    return df_pos

df_pos = fetch_gtfs_rt_positions()
if not df_pos.empty:
    display(df_pos.head())

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

# Wczytanie fizycznych lokalizacji przystanków
# Pobieramy pliki bezpośrednio z repozytorium GitHub
stops_file = "https://raw.githubusercontent.com/aszczi/Urban_mobility_in_Cracow/refs/heads/main/data/GTFS_ZTP_17.05.26/stops.txt"

if not df_pos.empty:
    try:
        df_stops = pd.read_csv(stops_file, dtype=str)
        # Konwersja coords na wartości numeryczne
        df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce").dropna()
        df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce").dropna()
        df_stops = df_stops.dropna(subset=['stop_lat', 'stop_lon']).reset_index(drop=True)
        
        # Zbudowanie drzewa KD dla szybkiego wyszukiwania najbliższego przystanku
        tree = cKDTree(df_stops[['stop_lat', 'stop_lon']].values)
        
        # Znalezienie najbliższego przystanku dla każdej pozycji pojazdu
        pos_coords = df_pos[['lat', 'lon']].values
        distances, indices = tree.query(pos_coords)
        
        # Przypisujemy nazwę najbliższego przystanku i jego pozycję do pojazdów
        df_pos['nearest_stop_name'] = df_stops.loc[indices, 'stop_name'].values
        df_pos['stop_lat'] = df_stops.loc[indices, 'stop_lat'].values
        df_pos['stop_lon'] = df_stops.loc[indices, 'stop_lon'].values
        
        # Grupujemy po przystanku, żeby obliczyć zagęszczenie
        # 'Zagęszczenie' pojazdów wykorzystamy jako sztuczne "opóźnienie" by wpasować się w model korków
        df_merged = df_pos.groupby(['nearest_stop_name', 'stop_lat', 'stop_lon', 'typ', 'hour'], as_index=False).agg(
            vehicles_count=('lat', 'count')
        )
        
        # Tworzymy wirtualne minuty opóźnienia na podstawie zagęszczenia (np. każdy pojazd powyżej 1 w tej samej okolicy to +2 min "korku")
        # Zakładamy, że >= 2 pojazdy to zator w ruchu miejskim.
        df_merged = df_merged[df_merged['vehicles_count'] >= 2].copy()
        df_merged['delay_min'] = df_merged['vehicles_count'] * 2.0 
        
        # Dopasowujemy nazwy kolumn do tego, czego oczekuje następna komórka 
        df_merged.rename(columns={'nearest_stop_name': 'stop_name'}, inplace=True)
        
        display(df_merged.sort_values(by='delay_min', ascending=False).head())
        print(f"Znaleziono {len(df_merged)} przystanków z wysokim zagęszczeniem pojazdów (lokalne zatory komunikacji miejskiej).")
    except Exception as e:
        print(f"Błąd podczas analizy lokalizacji względem przystanków: {e}")
        df_merged = pd.DataFrame()
else:
    print("Brak pojazdów w df_pos - upewnij się, ze dane GTFS-RT są dostępne.")
    df_merged = pd.DataFrame()

## Połączenie danych GTFS z danymi o ruchu samochodów (Traffic History)
Agregacja obydwu zbiorów w celu wskazania całościowego zatłoczenia w skali całego miasta (niezależnie czy opóźnione jest auto, czy autobus).

In [ ]:
# 1. Pobieranie danych o ruchu drogowym za pomocą API TomTom
import os
import requests
import pandas as pd
import datetime

# Jeśli chcesz podać własny klucz, przypisz go do zmiennej poniżej, np.:
# TOMTOM_API_KEY = "TWÓJ_KLUCZ"
TOMTOM_API_KEY = os.environ.get("TOMTOM_API_KEY", "ZW0nhcWdTcUE6SJjSJV5f9vXyLKbBHuR")

POINTS = [
    {"name": "Rondo Mogilskie", "lat": 50.0672, "lon": 19.9595},
    {"name": "Lubomirskiego", "lat": 50.0676, "lon": 19.9536},
    {"name": "Aleja Powstania Warszawskiego", "lat": 50.0647, "lon": 19.9608},
    {"name": "Grzegórzecka", "lat": 50.0578, "lon": 19.9564},
    {"name": "Rondo Grzegórzeckie", "lat": 50.0576, "lon": 19.9609},
    {"name": "Galeria Krakowska / Pawia", "lat": 50.0669, "lon": 19.9457},
    {"name": "Dworzec Główny Tunel", "lat": 50.0686, "lon": 19.9477},
    {"name": "Aleje Trzech Wieszczów - AGH", "lat": 50.0674, "lon": 19.9209},
    {"name": "Plac Inwalidów", "lat": 50.0671, "lon": 19.9269},
    {"name": "Nowy Kleparz", "lat": 50.0731, "lon": 19.9352},
    {"name": "Aleja Słowackiego / Łobzowska", "lat": 50.0692, "lon": 19.9302},
    {"name": "Cracovia / Aleja Mickiewicza", "lat": 50.0595, "lon": 19.9235},
    {"name": "Rondo Grunwaldzkie", "lat": 50.0492, "lon": 19.9317},
    {"name": "Most Dębnicki", "lat": 50.0530, "lon": 19.9278},
    {"name": "Rondo Matecznego", "lat": 50.0364, "lon": 19.9406},
    {"name": "Kalwaryjska", "lat": 50.0412, "lon": 19.9443},
    {"name": "Bonarka / Kamieńskiego", "lat": 50.0298, "lon": 19.9513},
    {"name": "Zakopiańska / Brożka", "lat": 50.0227, "lon": 19.9349},
    {"name": "Łagiewniki", "lat": 50.0202, "lon": 19.9378},
    {"name": "Opolska Estakada", "lat": 50.0857, "lon": 19.9536},
    {"name": "Rondo Polsadu", "lat": 50.0879, "lon": 19.9618},
    {"name": "Rondo Barei", "lat": 50.0911, "lon": 19.9742},
    {"name": "Aleja 29 Listopada / Opolska", "lat": 50.0861, "lon": 19.9520},
    {"name": "Aleja 29 Listopada / Dobrego Pasterza", "lat": 50.0953, "lon": 19.9746},
    {"name": "Rondo Ofiar Katynia", "lat": 50.0879, "lon": 19.8976},
    {"name": "Bronowice SKA", "lat": 50.0814, "lon": 19.8994},
    {"name": "Armii Krajowej / Zarzecze", "lat": 50.0738, "lon": 19.8985},
    {"name": "Czarnowiejska / Armii Krajowej", "lat": 50.0690, "lon": 19.9075},
    {"name": "Rondo Czyżyńskie", "lat": 50.0729, "lon": 20.0167},
    {"name": "Plac Centralny", "lat": 50.0720, "lon": 20.0372},
    {"name": "Aleja Pokoju / Centralna", "lat": 50.0667, "lon": 20.0028},
    {"name": "Nowohucka / Klimeckiego", "lat": 50.0503, "lon": 19.9765},
    {"name": "M1 / Aleja Pokoju", "lat": 50.0661, "lon": 20.0149},
    {"name": "Wielicka / Powstańców Wielkopolskich", "lat": 50.0335, "lon": 19.9621},
    {"name": "Estakada Obrońców Lwowa", "lat": 50.0396, "lon": 19.9627},
    {"name": "Bieżanowska / Wielicka", "lat": 50.0207, "lon": 19.9820},
    {"name": "Prokocim Szpital", "lat": 50.0117, "lon": 20.0005},
    {"name": "Węzeł Łagiewniki", "lat": 50.0136, "lon": 19.9325},
    {"name": "Węzeł Balice / A4", "lat": 50.0878, "lon": 19.7937},
    {"name": "Węzeł Tyniec / A4", "lat": 50.0196, "lon": 19.8078},
    {"name": "Węzeł Wielicka / A4", "lat": 50.0034, "lon": 20.0065}
]

def fetch_tomtom_flow(api_key, lat, lon):
    url = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/18/json"
    params = {"key": api_key, "point": f"{lat},{lon}", "unit": "kmph", "openLr": "false"}
    response = requests.get(url, params=params, timeout=15)
    if not response.ok:
        raise RuntimeError(f"TomTom HTTP {response.status_code}")
    return response.json().get("flowSegmentData")

rows = []
now = datetime.datetime.now()
timestamp_str = now.strftime('%Y-%m-%dT%H:00:00')

if TOMTOM_API_KEY:
    print(f"Pobieranie danych z API TomTom dla {len(POINTS)} punktów...")
    for point in POINTS:
        try:
            flow = fetch_tomtom_flow(TOMTOM_API_KEY, point["lat"], point["lon"])
            if flow:
                current_time = flow.get("currentTravelTime")
                free_flow = flow.get("freeFlowTravelTime")
                if current_time is not None and free_flow is not None:
                    delay_sec = max(0, current_time - free_flow)
                    rows.append({
                        "point_name": point["name"],
                        "lat": point["lat"],
                        "lon": point["lon"],
                        "delay_min": delay_sec / 60.0,
                        "hour": timestamp_str,
                        "typ": "Samochody"
                    })
        except Exception as e:
            pass # Ignoruj błędy dla pojedynczych punktów

print("Pobieranie statycznych danych z pliku CSV z GitHub.")
try:
    df_csv = pd.read_csv("https://raw.githubusercontent.com/aszczi/Urban_mobility_in_Cracow/refs/heads/main/traffic_history.csv")
    df_csv['timestamp'] = pd.to_datetime(df_csv['timestamp'])
    df_csv['hour'] = df_csv['timestamp'].dt.strftime('%Y-%m-%dT%H:00:00')
    df_csv['delay_min'] = df_csv['delay_sec'] / 60.0
    df_csv['typ'] = 'Samochody'
    df_csv_standard = df_csv[['point_name', 'lat', 'lon', 'delay_min', 'hour', 'typ']].copy()
    rows.extend(df_csv_standard.to_dict('records'))
except Exception as e:
    print(f"Nie udało się pobrać CSV: {e}")

df_cars_standard = pd.DataFrame(rows)

# 2. Przygotowujemy dane GTFS (Komunikacja Miejska) do tego samego formatu
if 'df_merged' in locals() and not df_merged.empty:
    df_gtfs_standard = df_merged[['stop_name', 'stop_lat', 'stop_lon', 'delay_min', 'hour', 'typ']].copy()
    df_gtfs_standard.rename(columns={'stop_name': 'point_name', 'stop_lat': 'lat', 'stop_lon': 'lon'}, inplace=True)
else:
    df_gtfs_standard = pd.DataFrame()

# 3. Złączenie obydwu zestawów w jeden olbrzymi rejestr opóźnień miejskich!
if not df_cars_standard.empty or not df_gtfs_standard.empty:
    df_all_traffic = pd.concat([df_cars_standard, df_gtfs_standard], ignore_index=True)
    df_all_traffic.dropna(subset=['delay_min', 'point_name'], inplace=True)
    if not df_all_traffic.empty:
        display(df_all_traffic.sample(min(5, len(df_all_traffic))))
    else:
        print("Brak pełnych danych (delay_min, point_name).")
else:
    df_all_traffic = pd.DataFrame()
    print("Brak jakichkolwiek danych (aut i GTFS).")

## Top najgorszych momentów i miejsc (Wykresy globalne)
Rzucamy okiem, o której godzinie miasto osiąga apogeum korków, oraz listujemy czołowe, najwęższe gardła komunikacyjne w systemie (niezależnie czy mówimy o przystanku czy samochodowym skrzyżowaniu).

In [ ]:
# WYKRES 1: Kiedy miasto jest najbardziej zatłoczone (sumarycznie uśrednione w podziale na godziny)
df_hourly = df_all_traffic.groupby('hour', as_index=False).agg(
    avg_delay_min=('delay_min', 'mean'),
    records=('delay_min', 'count')
).sort_values(by='hour')

fig_time = px.line(
    df_hourly, 
    x='hour', 
    y='avg_delay_min', 
    markers=True,
    title="Średnie zatłoczenie miasta Krakowa (opóźnienia w minutach) z podziałem na godziny",
    labels={'hour': 'Godzina', 'avg_delay_min': 'Średnie opóźnienie uczestnika ruchu (min)'}
)
fig_time.show()

# WYKRES 2: Jakie miejsca są najbardziej zatłoczone? - top 20
df_places = df_all_traffic.groupby('point_name', as_index=False).agg(
    avg_delay_min=('delay_min', 'mean'), 
    typ_glowny=('typ',  lambda x: x.mode()[0]) # przeważający typ w danym węźle
)

# Filtrujemy tylko te top 20 najgorszych lokalizacji
top_20_places = df_places.sort_values(by='avg_delay_min', ascending=False).head(20)

fig_places = px.bar(
    top_20_places,
    x='point_name',
    y='avg_delay_min',
    color='typ_glowny',
    title="Top 20 najbardziej opóźnionych / zatłoczonych punktów przestrzeni miejskiej",
    labels={'point_name': 'Nazwa skrzyżowania/przystanku', 'avg_delay_min': 'Średnie opóźnienie trwałego uczestnika (min)'},
    text_auto=':.1f',
    height=600
)
fig_places.update_layout(xaxis_tickangle=-45)
fig_places.show()

## Dynamiczna mapa Całościowego ruchu - Natężenie dróg Kolorem
Teraz rezygnujemy z poszukiwania zaledwie kilku autobusów a nakładamy cały zbiór punktów miejskich. Każdy z rekordów odszukuje optymalną ramkę krawędzi reprezentowanej ulicy, a całość wrzucamy na interaktywną mapę animowaną w czasie za pomocą GeoJSON. Zobaczymy na żywo jak malują się ulice!

In [ ]:
from IPython.display import display

if 'df_all_traffic' in locals() and not df_all_traffic.empty:
    
    # 1. Pobranie grafu drogowego miasta z OSMnx
    lokalizacja = "Kraków, Poland"
    print(f"Pobieranie geometrii dróg dla: {lokalizacja}... ")
    G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)
    
    # Do ucinania objętości weźmy uszeregowane "wycinki" dla Foliuma
    # Odchudzamy mapę o zduplikowane nazwy dla celów przypinania krawędzi (nearest_edges)
    unique_all_points = df_all_traffic.drop_duplicates(subset=['point_name', 'lat', 'lon']).copy()
    
    print("Przypinanie zatłoczonych miejsc (GTFS + Auta) do siatki ulic Krakowa (to zajmie kilkanaście sekund)...")
    lats = unique_all_points['lat'].values
    lons = unique_all_points['lon'].values
    
    # Wyszukujemy krawędzie dróg znajdujące się najbliżej punktów natężenia.
    try:
        nearest_edges = ox.nearest_edges(G, X=lons, Y=lats)
    except AttributeError:
        nearest_edges = ox.distance.nearest_edges(G, X=lons, Y=lats)
        
    point_to_edge = dict(zip(unique_all_points['point_name'], nearest_edges))
    
    # Przygotowanie palety kolorów
    features = []
    cmap = plt.get_cmap('RdYlGn_r') 
    norm = mcolors.Normalize(vmin=0, vmax=30)
    
    unique_hours = sorted(df_all_traffic['hour'].unique())
    print("Przetwarzanie opóźnień w przedziały godzinowe dla Folium...")
    
    for h in unique_hours:
        df_hour = df_all_traffic[df_all_traffic['hour'] == h]
        
        for idx, row in df_hour.iterrows():
            p_name = row['point_name']
            delay = row['delay_min']
            
            if p_name not in point_to_edge:
                continue
                
            edge = point_to_edge[p_name]
            u, v, key = edge
            
            coords = [[G.nodes[u]['x'], G.nodes[u]['y']], [G.nodes[v]['x'], G.nodes[v]['y']]]
            
            features.append({
                "type": "Feature",
                "geometry": {
                    "type": "LineString",
                    "coordinates": coords
                },
                "properties": {
                    "times": [h] * 2, 
                    "style": {
                        "color": mcolors.to_hex(cmap(norm(delay))),
                        "weight": 8,
                        "opacity": 0.85
                    }
                }
            })
            
    # Generowanie animowanej mapy 
    print("Tworzenie docelowej Interaktywnej Mapy z kolorowaniem ulic z całego miasta...")
    folium_map = folium.Map(location=[50.0614, 19.9383], zoom_start=13, tiles="cartodbdark_matter")
    
    TimestampedGeoJson(
        {"type": "FeatureCollection", "features": features},
        period="PT1H",
        add_last_point=False,
        auto_play=True,
        loop=True,
        max_speed=1,
        loop_button=True,
        time_slider_drag_update=True
    ).add_to(folium_map)
    
    display(folium_map)
else:
    print("Brak danych df_all_traffic do wyświetlenia mapy.")